# Domovina.tv — Canary 1B v2 BATCH transkripcija *(Colab)*

Batch transkripcija svih `.wav` datoteka uploadanih na Google Drive — generira `.canary.srt` i `.canary.csv` za svaki podcast. **Bez diarizacije** — diarizacija se radi odvojeno lokalno na Macu (`run_pipeline.sh --with-local-canary-diarize`) ili na Colabu (`colab_diarize/domovina_tv_diarize.ipynb`).

## Što ovaj notebook radi

```
Google Drive (canary_wav/)              Google Colab (G4 / L4 / T4)
┌────────────────────────────┐          ┌─────────────────────────────────┐
│ kanal_x/                   │  mount   │  transcribe_canary.py           │
│   ep001.wav        ───────────────►   │    nvidia/canary-1b-v2          │
│                            │          │    BF16 inference (2× brže)     │
│   ep001.wav.canary.srt ◄───────────── │    + .canary.csv                │
│   ep001.wav.canary.csv ◄───────────── │                                 │
└────────────────────────────┘          └─────────────────────────────────┘
```

Skripta je **idempotentna** — preskače WAV-ove koji već imaju `.canary.srt`. Možeš pokretati neograničeno; obrade se samo nove epizode.

## GPU tablica (testirano 2026-03-10)

| GPU | VRAM | units/h | sec/file | Napomena |
|---|---|---|---|---|
| **G4** (RTX PRO 6000 Blackwell) | **96 GB** | **8.71** | **~13s (BF16)** | Optimalno: najbrže + najjeftinije po fajlu |
| A100 80GB | 80 GB | 7.52 | ~70s | OK ali skuplje od G4 po fajlu |
| L4 | 24 GB | 1.71 | ~60s | Sporo, OOM na fajlovima >150 MB |
| T4 (free) | 16 GB | 1.67 | — | **OOM** za Canary 1B v2 (model ~19 GB FP32) |

**Preporuka**: G4 (Colab Pro+). T4 ne može učitati model — koristi L4 minimum, idealno G4.

## Kako koristiti

1. **GPU runtime**: Runtime → Change runtime type → `L4` (Pro) ili `G4` (Pro+).
2. **Drive folder**: WAV-ovi moraju biti u `MyDrive/domovina_fetch_data/canary_wav/{kanal}/...wav` (lokalni `run_pipeline.sh` ih tamo uploadira u koraku 2.5).
3. **Runtime → Run all** (⌘/Ctrl+F9). Prvi put restart kernela nakon instalacije — pokreni *Run all* još jednom.

## Procjena za 2545 podcasta

```
2545 fajlova × 13s (G4 BF16) = ~9.2 h
9.2 h × 8.71 units/h ≈ 80 units (Colab Pro+ ima ~500 units/mjesec)
```

---


## 0. Konfiguracija


In [ ]:
# ─── Drive locations ─────────────────────────────────────────────────────────
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_DATA_DIR    = "MyDrive/domovina_fetch_data/canary_wav"
INPUT_DIR         = f"{DRIVE_MOUNT_POINT}/{DRIVE_DATA_DIR}"

# ─── Ad-hoc transkripcija (opcionalno) ───────────────────────────────────────
# Zaseban Drive folder za WAV-ove koji NISU dio pipeline-a (predavanja, intervjui,
# random audio). Transkribira ih ista skripta u zasebnom batch-u (vidi cell 6.5).
# run_pipeline.sh rclone filter je hardkodiran na canary_wav/ — adhoc transkripti
# ostaju samo na Drive-u i ne dolaze na Mac kroz pipeline.
# Postavi None za skip; inače puni Drive path.
ADHOC_DRIVE_DIR   = "MyDrive/domovina_fetch_data/adhoc"   # ili None
ADHOC_DIR         = f"{DRIVE_MOUNT_POINT}/{ADHOC_DRIVE_DIR}" if ADHOC_DRIVE_DIR else None

# ─── Batch limits ────────────────────────────────────────────────────────────
LIMIT             = None   # int ili None — npr. 5 za testiranje, None za sve
DRY_RUN           = False  # True: samo prikaz, bez transkripcije

# ─── Jezik (Canary podržava multilingual + translation) ─────────────────────
SOURCE_LANG       = "hr"   # ISO kod izvornog jezika
TARGET_LANG       = "hr"   # isti kao source = transkripcija (bez prijevoda)

# ─── Auto-shutdown ───────────────────────────────────────────────────────────
# True: nakon završetka batcha gasi Colab instancu (runtime.unassign) da
# odmah stane trošenje compute units. Postavi na False kad debugiraš ili
# želiš ručno pregledati output prije gašenja.
AUTO_SHUTDOWN     = True

# ─── Repo ────────────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/domovinatv/fetch.domovina.tv.git"
REPO_PATH = "/content/fetch.domovina.tv"

print("Konfiguracija učitana.")
print(f"  Input:         {INPUT_DIR}")
print(f"  Ad-hoc:        {ADHOC_DIR if ADHOC_DIR else '(disabled)'}")
print(f"  Limit:         {LIMIT if LIMIT else 'sve'}")
print(f"  Dry run:       {DRY_RUN}")
print(f"  Lang:          {SOURCE_LANG} → {TARGET_LANG}")
print(f"  Auto-shutdown: {AUTO_SHUTDOWN}")


## 1. GPU provjera

Canary 1B v2 model zauzima ~19 GB FP32 ili ~10 GB BF16. **T4 nije dovoljan** — koristi L4 (24 GB) ili G4 (96 GB).


In [ ]:
import subprocess
out = subprocess.check_output(["nvidia-smi", "-L"]).decode().strip()
print(out)

mem = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total,memory.free", "--format=csv,noheader,nounits"]
).decode().strip()
total, free = [int(x.strip()) for x in mem.split(",")]
print(f"\nVRAM: {free} MB slobodno / {total} MB ukupno ({free/total*100:.0f}%)")

if total < 22000:
    print(f"\nUPOZORENJE: Manje od 22 GB VRAM ({total} MB) — Canary 1B v2 može OOM.")
    print("Idi na Runtime → Change runtime type → L4 (Pro) ili G4 (Pro+).")


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount(DRIVE_MOUNT_POINT)

import os
assert os.path.isdir(INPUT_DIR), (
    f"Direktorij ne postoji: {INPUT_DIR}\n"
    f"Provjeri DRIVE_DATA_DIR ili da li su WAV-ovi uploadani."
)
n_entries = len(os.listdir(INPUT_DIR))
print(f"Drive mountan. INPUT_DIR sadrži: {n_entries} entry-ja (kanala/foldera).")


## 3. Clone / pull repo


In [ ]:
import os, subprocess

if not os.path.isdir(REPO_PATH):
    print(f"Kloniram repo u {REPO_PATH}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    print("Repo postoji, povlačim najnovije...")
    subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], check=True)

TRANSCRIBE_SCRIPT = f"{REPO_PATH}/colab_canary/transcribe_canary.py"
assert os.path.isfile(TRANSCRIBE_SCRIPT), f"Nedostaje skripta: {TRANSCRIBE_SCRIPT}"
print(f"Skripta spremna: {os.path.basename(TRANSCRIBE_SCRIPT)}")


## 4. Instalacija dependencija (~2 min)

Instalira NeMo toolkit (sadrži Canary). Prvi put prisilno gasi kernel kako bi novi numpy/scipy proradili (inače `ImportError: _center` u NeMo). Nakon restarta klikni **Runtime → Run all** ponovno.


In [ ]:
import os

MARKER = "/content/.canary_deps_ok"

def _have_deps():
    try:
        import nemo.collections.asr  # noqa: F401
        return True
    except Exception as e:
        print(f"deps check: {type(e).__name__}: {e}")
        return False

if os.path.exists(MARKER) and _have_deps():
    print("Dependencies već instalirani i spremni.")
else:
    print("Instaliram NeMo (sadrži Canary) — traje ~2 min...")
    get_ipython().system(
        'pip install -qU numpy "nemo_toolkit[asr]" 2>&1 | tail -5'
    )
    open(MARKER, "w").write("ok")
    print("")
    print("=" * 60)
    print(" Instalacija gotova — gasim kernel radi čistog reloada.")
    print(" → Nakon restarta klikni Runtime → Run all ponovno.")
    print("=" * 60)
    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)


In [ ]:
# Bezopasni shimovi za starije NeMo import path-eve
import sys
class _DummyYTTM: pass
sys.modules.setdefault("youtokentome", _DummyYTTM)

import huggingface_hub
if getattr(huggingface_hub, "ModelFilter", None) is None:
    class ModelFilter: pass
    huggingface_hub.ModelFilter = ModelFilter

import torch
print("torch:", torch.__version__,
      "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
print("BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)


## 5. Pregled posla (dry run)

Skenira Drive i prikazuje koliko WAV-ova je već transkribirano vs. koliko će se obraditi u ovom runu. Ne radi nikakvu obradu.


In [ ]:
def scan_progress(input_dir):
    wavs, with_srt = [], []
    for root, _, files in os.walk(input_dir, followlinks=True):
        for f in files:
            if f.startswith("._") or not f.endswith(".wav"):
                continue
            p = os.path.join(root, f)
            wavs.append(p)
            if os.path.exists(p + ".canary.srt"):
                with_srt.append(p)
    return wavs, with_srt

print("Skeniram Drive (može potrajati ~30s za veliki korpus)...")
wavs, with_srt = scan_progress(INPUT_DIR)
to_transcribe = [w for w in wavs if not os.path.exists(w + ".canary.srt")]

print(f"\n  Ukupno WAV datoteka:     {len(wavs)}")
print(f"  Već transkribirano:      {len(with_srt)} ({len(with_srt)/max(len(wavs),1)*100:.1f}%)")
print(f"  Za transkripciju (novo): {len(to_transcribe)}")

if LIMIT:
    print(f"\n  LIMIT={LIMIT} → obradit će se najviše {LIMIT} datoteka")

# ETA po GPU-u
gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ""
if "rtx pro 6000" in gpu_name or "g4" in gpu_name:
    sec_per_file = 13
elif "a100" in gpu_name:
    sec_per_file = 25
elif "l4" in gpu_name:
    sec_per_file = 60
else:
    sec_per_file = 90
n_to_run = min(LIMIT or 10**9, len(to_transcribe))
if n_to_run:
    eta_min = n_to_run * sec_per_file / 60
    print(f"\n  Procjena trajanja na ovom GPU-u: ~{eta_min:.0f} min")
    print(f"  (heuristika: ~{sec_per_file}s/file × {n_to_run} file × BF16)")


## 6. Pokreni batch transkripciju

Poziva `transcribe_canary.py` koji:
- Učita Canary 1B v2 (BF16 ako GPU podržava → ~10 GB VRAM, 2× brže od FP32)
- Skenira `INPUT_DIR` rekurzivno za `.wav` datoteke
- Preskače sve koje već imaju `.canary.srt` (idempotentno)
- Generira `.canary.srt` i `.canary.csv` u istom folderu kao WAV (na Drive-u)
- Heartbeat svakih 60s tijekom dugačkih datoteka

Output ide odmah na Drive — sljedeći put kad pokreneš `run_pipeline.sh` lokalno, korak 0 (rclone) povuče nove `.canary.srt` na Mac.


In [ ]:
import shlex

cmd = [
    "python", "-u", TRANSCRIBE_SCRIPT,
    "--input-dir", INPUT_DIR,
    "--source-lang", SOURCE_LANG,
    "--target-lang", TARGET_LANG,
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if DRY_RUN:
    cmd += ["--dry-run"]

print(">", " ".join(shlex.quote(c) for c in cmd))
print()
get_ipython().system(" ".join(shlex.quote(c) for c in cmd))


## 6.5 Ad-hoc transkripcija (opcionalno)

Ako je `ADHOC_DIR` postavljen u cell 0, transkribira WAV-ove iz zasebnog Drive foldera **izvan glavnog pipeline-a** (predavanja, intervjui, ad-hoc audio koji nije podcast epizoda).

**Izolacija od pipeline-a:**
- `run_pipeline.sh` rclone filter je hardkodiran na `canary_wav/` poddir → ad-hoc transkripti NE dolaze na Mac kroz pipeline.
- `count_progress.js`, RAG skripte i ostalo skeniraju samo lokalni `storage/output/` → ad-hoc file-ovi ni ne postoje u njihovom svjetonazoru.
- Naming format ad-hoc WAV-ova nije bitan (ne mora biti `YYYYMMDD_title_yt_XXX.wav`).

**Kako dohvatiti ad-hoc transkripte lokalno:**

```bash
rclone copy google_drive_ms:domovina_fetch_data/adhoc ~/adhoc_transkripti \
  --filter "+ *.canary.*" --filter "- *" --drive-shared-with-me --progress
```

Postavi `ADHOC_DRIVE_DIR = None` u cell 0 za skip.


In [ ]:
import shlex, os

if not ADHOC_DIR:
    print("ADHOC_DIR=None → preskačem ad-hoc transkripciju.")
elif not os.path.isdir(ADHOC_DIR):
    print(f"⚠️  Ad-hoc direktorij ne postoji: {ADHOC_DIR}")
    print(f"    Kreiraj ga na Drive-u i dropaj WAV-ove (Drive web/desktop client).")
    print(f"    Ovaj run preskače ad-hoc fazu. Postavi ADHOC_DRIVE_DIR=None ako ne planiraš koristiti.")
else:
    # Pre-flight: koliko WAV-ova već ima .canary.srt
    adhoc_wavs, adhoc_done = [], []
    for root, _, files in os.walk(ADHOC_DIR, followlinks=True):
        for f in files:
            if f.startswith("._") or not f.endswith(".wav"):
                continue
            p = os.path.join(root, f)
            adhoc_wavs.append(p)
            if os.path.exists(p + ".canary.srt"):
                adhoc_done.append(p)
    adhoc_pending = len(adhoc_wavs) - len(adhoc_done)

    print(f"Ad-hoc batch: INPUT_DIR = {ADHOC_DIR}")
    print(f"  WAV ukupno:              {len(adhoc_wavs)}")
    print(f"  Već transkribirano:      {len(adhoc_done)}")
    print(f"  Za transkripciju (novo): {adhoc_pending}")
    print()

    if adhoc_pending == 0:
        print("✓ Ništa za obraditi u ad-hoc batch-u.")
    else:
        cmd = [
            "python", "-u", TRANSCRIBE_SCRIPT,
            "--input-dir", ADHOC_DIR,
            "--source-lang", SOURCE_LANG,
            "--target-lang", TARGET_LANG,
        ]
        if LIMIT:
            cmd += ["--limit", str(LIMIT)]
        if DRY_RUN:
            cmd += ["--dry-run"]
        print(">", " ".join(shlex.quote(c) for c in cmd))
        print()
        get_ipython().system(" ".join(shlex.quote(c) for c in cmd))


## 7. Sažetak — što je novo na Drive-u

Delta novih `.canary.srt` u ovom runu. Ovaj broj je input za diarizaciju (sljedeći Colab notebook ili lokalno).


In [ ]:
wavs2, with_srt2 = scan_progress(INPUT_DIR)
new_srt = len(with_srt2) - len(with_srt)

print("─" * 60)
print(f"  Novih .canary.srt:  +{new_srt}")
print("─" * 60)
print(f"  Stanje na Drive-u:")
print(f"    {len(wavs2)} WAV ukupno")
print(f"    {len(with_srt2)}/{len(wavs2)} transkribirano ({len(with_srt2)/max(len(wavs2),1)*100:.1f}%)")
print()
print("Sljedeći koraci:")
print("  1. Diarizacija: otvori colab_diarize/domovina_tv_diarize.ipynb")
print("  2. Lokalno: ./run_pipeline.sh (korak 0 rclone povuče nove .canary.srt)")


## 8. Auto-shutdown — pusti Colab instancu

Ako je `AUTO_SHUTDOWN = True` (cell 0), poziva `runtime.unassign()` koji **odmah** terminira VM i prestaje trošiti compute units. Bez ovoga instanca živi do Colab idle timeouta (~90 min) tijekom kojeg credits i dalje cure.

Postavi `AUTO_SHUTDOWN = False` na vrhu kad debugiraš ili kad želiš ručno pregledati output prije nego što se sesija ugasi.


In [ ]:
if AUTO_SHUTDOWN:
    print("AUTO_SHUTDOWN=True → gasim Colab runtime za 5s…")
    print("(Ako želiš odustati: Runtime → Interrupt execution sada.)")
    import time
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
else:
    print("AUTO_SHUTDOWN=False → instanca ostaje živa.")
    print("Ručno gašenje: Runtime → Disconnect and delete runtime")
    print("(ili pokreni: from google.colab import runtime; runtime.unassign())")
